In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Wed Oct 28 09:23:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
import torch
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print(
        f"  Memory: "
        f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB"
    )

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
  Memory: 14.56 GiB
GPU 1: Tesla T4
  Memory: 14.56 GiB


In [3]:
!pip install -q uv
!uv venv --python 3.12 --seed /kaggle/working/ckv-venv
!uv pip install --python /kaggle/working/ckv-venv/bin/python vllm --torch-backend=auto

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment with seed packages at: ckv-venv
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + pip==26.2.2
Activate with: source ckv-venv/bin/activate
Using Python 3.12.3 environment at: ckv-venv
Resolved 198 packages in 5.09s                                       
Prepared 198 packages in 52.19s                                          
░░░░░░░░░░░░░░░░░░░░ [0/198] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Ins

In [4]:
!/kaggle/working/ckv-venv/bin/python -c "import torch, vllm; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('vLLM:', vllm.__version__)"

PyTorch: 2.13.0+cu132
CUDA: True
vLLM: 0.32.0


In [7]:
%%writefile /kaggle/working/test_vllm.py
import torch
from vllm import LLM, SamplingParams


def main():
    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0))

    llm = LLM(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        tensor_parallel_size=1,
        max_model_len=2048,
        gpu_memory_utilization=0.80,
        trust_remote_code=False,
    )

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=100,
    )

    prompts = [
        "Explain why tenant isolation matters in multi-tenant LLM inference.",
        "What is a KV cache in a transformer model?",
    ]

    outputs = llm.generate(prompts, sampling_params)

    for i, output in enumerate(outputs, 1):
        print(f"\n--- Response {i} ---")
        print(output.outputs[0].text)

    print("\nSUCCESS: vLLM inference completed.")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/test_vllm.py


In [8]:
!/kaggle/working/ckv-venv/bin/python /kaggle/working/test_vllm.py

CUDA available: True
GPU: Tesla T4
(MainProcess pid=294) INFO 10-28 09:43:36 [api_utils.py:286] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=294) INFO 10-28 09:43:37 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=294) WARNING 10-28 09:43:37 [model.py:2439] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
(MainProcess pid=294) WARNING 10-28 09:43:37 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=294) INFO 10-28 09:43:37 [model.py:2152] Using max model len 2048
(MainProcess pid=294) INFO 10-28 09:43:37 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=294) INFO 10-28 09:43:39 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_

In [11]:
%cd /kaggle/working
!git clone https://github.com/aditya3275/conf-kv-1.git
%cd /kaggle/working/conf-kv-1
!git status --short
!ls -la

/kaggle/working
Cloning into 'conf-kv-1'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 128 (delta 25), reused 119 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 63.98 KiB | 1.52 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/kaggle/working/conf-kv-1
total 60
drwxr-xr-x 10 root root 4096 Oct 28 10:25 .
drwxr-xr-x  5 root root 4096 Oct 28 10:25 ..
drwxr-xr-x 12 root root 4096 Oct 28 10:25 app
drwxr-xr-x  2 root root 4096 Oct 28 10:25 configs
drwxr-xr-x  2 root root 4096 Oct 28 10:25 docs
-rw-r--r--  1 root root  331 Oct 28 10:25 .env.example
drwxr-xr-x  8 root root 4096 Oct 28 10:25 .git
-rw-r--r--  1 root root  143 Oct 28 10:25 .gitignore
drwxr-xr-x  3 root root 4096 Oct 28 10:25 policies
-rw-r--r--  1 root root   74 Oct 28 10:25 pytest.ini
-rw-r--r--  1 root root  182 Oct 28 10:25 README.md
-rw-r--r--  1 root root  224 Oct 28 10:25 requirements.txt
d

In [12]:
%cd /kaggle/working/conf-kv-1

!cat README.md
!cat requirements.txt
!find app verifkv_service tests scripts configs -maxdepth 3 -type f


/kaggle/working/conf-kv-1
cat > README.md <<'EOF'
# Conf-kv-1

ConfidentialKV-Scheduler

External policy control plane for multi-tenant LLM inference.

## Current Phase

Phase 1 — Foundation + Contracts
EOFfastapi>=0.115,<1.0
uvicorn[standard]>=0.34,<1.0

pydantic>=2.10,<3.0
pydantic-settings>=2.7,<3.0

httpx>=0.28,<1.0

aiosqlite>=0.20,<1.0

PyYAML>=6.0,<7.0

pytest>=8.3,<9.0
pytest-asyncio>=0.25,<1.0

python-dotenv>=1.0,<2.0app/scheduler/scheduler.py
app/scheduler/__init__.py
app/kv/__init__.py
app/common/__init__.py
app/common/logging.py
app/common/request_context.py
app/main.py
app/registry/tenant_registry.py
app/registry/health_monitor.py
app/registry/__init__.py
app/registry/backend_registry.py
app/opa/client.py
app/opa/__init__.py
app/__init__.py
app/models/verifkv.py
app/models/scheduling.py
app/models/backend.py
app/models/cache.py
app/models/opa.py
app/models/__init__.py
app/models/tenant.py
app/models/routing.py
app/models/inference.py
app/models/enums.py
app/models/lifecycl

In [13]:
!nvidia-smi

Wed Oct 28 10:26:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
%cd /kaggle/working/conf-kv-1

print("===== MAIN APPLICATION =====")
!sed -n '1,240p' app/main.py

print("\n===== SETTINGS =====")
!sed -n '1,240p' app/config/settings.py

print("\n===== BACKEND CONFIGURATION =====")
!cat configs/backends.yaml

print("\n===== ROOT FILES =====")
!find . -maxdepth 2 -type f \
  -not -path './.git/*' \
  | sort

/kaggle/working/conf-kv-1
===== MAIN APPLICATION =====
from __future__ import annotations

import logging
import time
from contextlib import asynccontextmanager
from datetime import datetime, timezone
from uuid import UUID, uuid4

import httpx
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.responses import Response

from app.common.logging import configure_logging
from app.common.request_context import (
    get_request_id,
    new_request_id,
    reset_request_id,
    set_request_id,
)
from app.config.settings import settings
from app.enforcement.cache_policy import build_cache_policy
from app.enforcement.proxy import EnforcementProxy
from app.enforcement.router import EnforcementRouter, RoutingError
from app.gateway.auth import make_authenticator
from app.models.inference import ClientInferenceRequest
from app.models.opa import OPARequest
from app.opa.client import (
    OPAClient,
    OPADecisionError,
    OPAUnavailableError,
)
from app.registry.b

In [15]:
import os
import secrets

os.environ["VERIFKV_HMAC_SECRET"] = secrets.token_hex(32)

print("HMAC secret configured for this notebook session.")

HMAC secret configured for this notebook session.


In [16]:
%cd /kaggle/working/conf-kv-1

!grep -nE "FastAPI|lifespan|startup|include_router|mount|OPA|VERIFKV|backend|health|ready" app/main.py
!grep -nE "BaseSettings|VERIFKV_HMAC_SECRET|OPA|VERIFKV|DATABASE|TIMEOUT" app/config/settings.py
!find . -maxdepth 3 -type f \
  \( -iname '*opa*' -o -iname '*.rego' -o -iname '*docker*' -o -iname '*compose*' \) \
  -not -path './.git/*' | sort

/kaggle/working/conf-kv-1
10:from fastapi import Depends, FastAPI, Header, HTTPException, Request
26:from app.models.opa import OPARequest
28:    OPAClient,
29:    OPADecisionError,
30:    OPAUnavailableError,
32:from app.registry.backend_registry import BackendRegistry
33:from app.registry.health_monitor import BackendHealthMonitor
59:        self.backend_error_count = 0
89:                "# TYPE confidentialkv_backend_errors_total counter",
90:                f"confidentialkv_backend_errors_total {self.backend_error_count}",
115:async def lifespan(app: FastAPI):
122:    backend_registry = BackendRegistry(settings.control_plane_db_path)
124:    opa_client = OPAClient(
152:    # Backend health monitoring
155:    health_monitor = BackendHealthMonitor(
156:        registry=backend_registry,
157:        interval_seconds=settings.health_cache_ttl_seconds,
158:        timeout_seconds=settings.health_check_timeout_seconds,
183:        await backend_registry.initialize()
195:        # Backen

In [17]:
%cd /kaggle/working/conf-kv-1

!cat requirements.txt

/kaggle/working/conf-kv-1
fastapi>=0.115,<1.0
uvicorn[standard]>=0.34,<1.0

pydantic>=2.10,<3.0
pydantic-settings>=2.7,<3.0

httpx>=0.28,<1.0

aiosqlite>=0.20,<1.0

PyYAML>=6.0,<7.0

pytest>=8.3,<9.0
pytest-asyncio>=0.25,<1.0

python-dotenv>=1.0,<2.0

In [18]:
!/kaggle/working/ckv-venv/bin/python -m pip install \
  "fastapi>=0.115,<1.0" \
  "uvicorn[standard]>=0.34,<1.0" \
  "pydantic>=2.10,<3.0" \
  "pydantic-settings>=2.7,<3.0" \
  "httpx>=0.28,<1.0" \
  "aiosqlite>=0.20,<1.0" \
  "PyYAML>=6.0,<7.0" \
  "python-dotenv>=1.0,<2.0"

In [19]:
%cd /kaggle/working/conf-kv-1
!sed -n '1,240p' policies/rego/authz.rego

/kaggle/working/conf-kv-1
package confidentialkv

import rego.v1


default allow := false


# ---------------------------------------------------------
# Main authorization decision
# ---------------------------------------------------------

allow if {
    input.tenant.active
    input.tenant.quota_remaining > 0

    input.recommendation.backend_id == input.resource.backend_id

    input.recommendation.zone in input.tenant.allowed_zones

    input.resource.configured_status == "available"
    input.resource.health_status == "healthy"
    input.resource.capacity_available

    high_security_requirement_satisfied
}


high_security_requirement_satisfied if {
    input.tenant.security_level == "standard"
}


high_security_requirement_satisfied if {
    input.tenant.security_level == "high"
    input.recommendation.zone == "secure"
}


# ---------------------------------------------------------
# Human-readable reason
# ---------------------------------------------------------

reason := "

In [20]:
!which opa || true
!opa version 2>/dev/null || true

In [21]:
%cd /kaggle/working/conf-kv-1

!curl -fsSL \
  https://openpolicyagent.org/downloads/latest/opa_linux_amd64 \
  -o /kaggle/working/opa

!chmod +x /kaggle/working/opa
!/kaggle/working/opa version

/kaggle/working/conf-kv-1
Version: 1.21.4
Build Commit: 05c845177c80747a2aabb607f9318f981993cc76-dirty
Build Timestamp: 2026-10-23T13:00:56Z
Build Hostname: 
Go Version: go1.27.3
Platform: linux/amd64
Rego Version: v1
WebAssembly: available


In [22]:
%cd /kaggle/working/conf-kv-1

!/kaggle/working/opa check policies/rego
!/kaggle/working/opa test policies/rego -v

/kaggle/working/conf-kv-1
1 error occurred during loading: policies/rego/authz.rego:61: rego_parse_error: unexpected not keyword: expected \n or ; or }
	    input.recommendation.zone not in input.tenant.allowed_zones
	                              ^
1 error occurred during loading: policies/rego/authz.rego:61: rego_parse_error: unexpected not keyword: expected \n or ; or }
	    input.recommendation.zone not in input.tenant.allowed_zones
	                              ^


In [24]:
%cd /kaggle/working/conf-kv-1

from pathlib import Path

p = Path("policies/rego/authz.rego")
s = p.read_text()

old = "input.recommendation.zone not in input.tenant.allowed_zones"
new = "not (input.recommendation.zone in input.tenant.allowed_zones)"

assert s.count(old) == 1, "Expected exactly one invalid expression"
p.write_text(s.replace(old, new))

print("Fixed the invalid membership expression.")

/kaggle/working/conf-kv-1
Fixed the invalid membership expression.


In [25]:
%cd /kaggle/working/conf-kv-1

!/kaggle/working/opa check policies/rego
!/kaggle/working/opa test policies/rego -v

/kaggle/working/conf-kv-1
policies/rego/authz_test.rego:114:
data.confidentialkv.test_unhealthy_backend_denied: PASS (3.439829ms)
policies/rego/authz_test.rego:6:
data.confidentialkv.test_high_security_secure_backend_allowed: PASS (3.551641ms)
policies/rego/authz_test.rego:87:
data.confidentialkv.test_exhausted_quota_denied: PASS (3.778349ms)
policies/rego/authz_test.rego:168:
data.confidentialkv.test_backend_mismatch_denied: PASS (5.195481ms)
policies/rego/authz_test.rego:33:
data.confidentialkv.test_high_security_general_backend_denied: PASS (2.596758ms)
policies/rego/authz_test.rego:60:
data.confidentialkv.test_inactive_tenant_denied: PASS (1.755351ms)
policies/rego/authz_test.rego:141:
data.confidentialkv.test_capacity_unavailable_denied: PASS (2.244053ms)
--------------------------------------------------------------------------------
PASS: 7/7


In [26]:
import subprocess
import time
import requests

opa_bin = "/kaggle/working/opa"
policy_dir = "/kaggle/working/conf-kv-1/policies/rego"

opa_log = open("/kaggle/working/opa.log", "w")

opa_process = subprocess.Popen(
    [
        opa_bin, "run",
        "--server",
        "--addr=127.0.0.1:8181",
        policy_dir,
    ],
    stdout=opa_log,
    stderr=subprocess.STDOUT,
)

for _ in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:8181/health",
            timeout=1,
        )
        if response.ok:
            print("OPA is running:", response.status_code)
            break
    except requests.RequestException:
        time.sleep(1)
else:
    print("OPA did not become healthy. Recent logs:")
    print(open("/kaggle/working/opa.log").read()[-4000:])

OPA is running: 200


In [27]:
%cd /kaggle/working/conf-kv-1

!sed -n '1,240p' verifkv_service/main.py
!sed -n '1,260p' verifkv_service/app.py
!sed -n '1,220p' app/main.py

/kaggle/working/conf-kv-1
import uvicorn

if __name__ == "__main__":
    uvicorn.run(
        "verifkv_service.app:app",
        host="127.0.0.1",
        port=8100,
        reload=False,
    )
from __future__ import annotations

from contextlib import asynccontextmanager

from fastapi import FastAPI, status

from app.models.verifkv import VerifKVEvent

from verifkv_service.store import VerifKVStore
from verifkv_service.watchdog import (
    BackendWatchdog,
)
from verifkv_service.writer import (
    VerifKVWriter,
)


class VerifKVApplicationState:

    def __init__(
        self,
        db_path: str,
        hmac_secret: str,
        queue_maxsize: int,
        watch_interval_seconds: float,
        watch_timeout_seconds: float,
    ) -> None:

        self.store = VerifKVStore(
            db_path=db_path,
            hmac_secret=hmac_secret,
        )

        self.writer = VerifKVWriter(
            store=self.store,
            queue_maxsize=queue_maxsize,
        )

        sel

In [28]:
import os
import secrets
from pathlib import Path

PROJECT = Path("/kaggle/working/conf-kv-1")
DATA = PROJECT / "data"
DATA.mkdir(parents=True, exist_ok=True)

# Shared by the gateway and VerifKV in this notebook session.
if "VERIFKV_HMAC_SECRET" not in os.environ:
    os.environ["VERIFKV_HMAC_SECRET"] = secrets.token_hex(32)

os.environ["VERIFKV_URL"] = "http://127.0.0.1:8100"
os.environ["CONTROL_PLANE_DB_PATH"] = str(DATA / "control_plane.db")
os.environ["VERIFKV_DB_PATH"] = str(DATA / "verifkv.db")

print("Shared HMAC secret configured.")
print("Control-plane DB:", os.environ["CONTROL_PLANE_DB_PATH"])
print("VerifKV DB:", os.environ["VERIFKV_DB_PATH"])
print("VerifKV URL:", os.environ["VERIFKV_URL"])

Shared HMAC secret configured.
Control-plane DB: /kaggle/working/conf-kv-1/data/control_plane.db
VerifKV DB: /kaggle/working/conf-kv-1/data/verifkv.db
VerifKV URL: http://127.0.0.1:8100


In [29]:
%cd /kaggle/working/conf-kv-1

!sed -n '1,240p' verifkv_service/app.py
!sed -n '1,200p' verifkv_service/main.py
!sed -n '1,240p' verifkv_service/store.py

/kaggle/working/conf-kv-1
from __future__ import annotations

from contextlib import asynccontextmanager

from fastapi import FastAPI, status

from app.models.verifkv import VerifKVEvent

from verifkv_service.store import VerifKVStore
from verifkv_service.watchdog import (
    BackendWatchdog,
)
from verifkv_service.writer import (
    VerifKVWriter,
)


class VerifKVApplicationState:

    def __init__(
        self,
        db_path: str,
        hmac_secret: str,
        queue_maxsize: int,
        watch_interval_seconds: float,
        watch_timeout_seconds: float,
    ) -> None:

        self.store = VerifKVStore(
            db_path=db_path,
            hmac_secret=hmac_secret,
        )

        self.writer = VerifKVWriter(
            store=self.store,
            queue_maxsize=queue_maxsize,
        )

        self.watchdog = BackendWatchdog(
            writer=self.writer,
            interval_seconds=(watch_interval_seconds),
            timeout_seconds=(watch_timeout_seconds)

In [30]:
import os
import subprocess
import time
import requests

PROJECT = "/kaggle/working/conf-kv-1"
PYTHON = "/kaggle/working/ckv-venv/bin/python"

assert os.environ.get("VERIFKV_HMAC_SECRET"), (
    "Shared HMAC secret is missing. Recreate it before starting services."
)

# Set these before starting the service.
os.environ["VERIFKV_URL"] = "http://127.0.0.1:8100"
os.environ["VERIFKV_DB_PATH"] = f"{PROJECT}/data/verifkv.db"

# Stop an earlier service process if this cell was already run.
if "verifkv_process" in globals() and verifkv_process.poll() is None:
    verifkv_process.terminate()
    verifkv_process.wait(timeout=5)

verifkv_log = open("/kaggle/working/verifkv.log", "w")

verifkv_process = subprocess.Popen(
    [
        PYTHON, "-c",
        """
import os
import uvicorn
from verifkv_service.app import create_app

app = create_app(
    db_path=os.environ["VERIFKV_DB_PATH"],
    hmac_secret=os.environ["VERIFKV_HMAC_SECRET"],
)

uvicorn.run(app, host="127.0.0.1", port=8100, log_level="info")
""",
    ],
    cwd=PROJECT,
    env=os.environ.copy(),
    stdout=verifkv_log,
    stderr=subprocess.STDOUT,
)

for _ in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:8100/health",
            timeout=1,
        )
        if response.ok:
            print("VerifKV healthy:", response.json())
            break
    except requests.RequestException:
        time.sleep(1)
else:
    print("VerifKV failed to start. Logs:")
    print(open("/kaggle/working/verifkv.log").read()[-5000:])

VerifKV healthy: {'status': 'ok', 'service': 'verifkv'}


In [31]:
import requests

for name, url in [
    ("OPA", "http://127.0.0.1:8181/health"),
    ("VerifKV", "http://127.0.0.1:8100/health"),
]:
    try:
        r = requests.get(url, timeout=3)
        print(name, r.status_code, r.text[:300])
    except requests.RequestException as exc:
        print(name, "ERROR:", exc)

OPA 200 {}

VerifKV 200 {"status":"ok","service":"verifkv"}


In [32]:
%cd /kaggle/working/conf-kv-1

print("===== BACKEND REGISTRY SCRIPT =====")
!sed -n '1,240p' scripts/register_backend.py

print("\n===== BACKEND CONFIG =====")
!cat configs/backends.yaml

print("\n===== GATEWAY SETTINGS =====")
!sed -n '1,260p' app/config/settings.py

print("\n===== BACKEND REGISTRY =====")
!sed -n '1,280p' app/registry/backend_registry.py

print("\n===== ENFORCEMENT PROXY =====")
!sed -n '1,260p' app/enforcement/proxy.py

/kaggle/working/conf-kv-1
===== BACKEND REGISTRY SCRIPT =====
from __future__ import annotations

import argparse
import asyncio

from app.config.settings import get_settings
from app.models.backend import Backend
from app.models.enums import BackendStatus, Zone
from app.registry.backend_registry import BackendRegistry


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Register a ConfidentialKV serving backend"
    )

    parser.add_argument(
        "--id",
        required=True,
        dest="backend_id",
    )

    parser.add_argument(
        "--endpoint",
        required=False,
        default=None,
    )

    parser.add_argument(
        "--zone",
        required=True,
        choices=[zone.value for zone in Zone],
    )

    parser.add_argument(
        "--gpu",
        type=int,
        nargs="*",
        default=[],
    )

    parser.add_argument(
        "--available",
        action="store_true",
    )

    return pa

In [33]:
import os
import subprocess
import time
import requests

PYTHON = "/kaggle/working/ckv-venv/bin/python"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Start vLLM on a separate port from OPA and VerifKV.
if "vllm_api_process" in globals() and vllm_api_process.poll() is None:
    print("vLLM API server is already running.")
else:
    vllm_api_log = open("/kaggle/working/vllm_api.log", "w")

    vllm_api_process = subprocess.Popen(
        [
            PYTHON, "-m", "vllm.entrypoints.openai.api_server",
            "--model", MODEL,
            "--host", "127.0.0.1",
            "--port", "8001",
            "--max-model-len", "2048",
            "--gpu-memory-utilization", "0.70",
            "--tensor-parallel-size", "1",
        ],
        stdout=vllm_api_log,
        stderr=subprocess.STDOUT,
        cwd="/kaggle/working/conf-kv-1",
        env=os.environ.copy(),
    )

    print("Started vLLM API server; PID:", vllm_api_process.pid)

for _ in range(90):
    if vllm_api_process.poll() is not None:
        print("vLLM exited. Recent log:")
        print(open("/kaggle/working/vllm_api.log").read()[-6000:])
        break

    try:
        r = requests.get("http://127.0.0.1:8001/v1/models", timeout=2)
        if r.ok:
            print("vLLM API ready:", r.status_code)
            print(r.json())
            break
    except requests.RequestException:
        pass

    time.sleep(2)
else:
    print("vLLM is still starting. Recent log:")
    print(open("/kaggle/working/vllm_api.log").read()[-4000:])

Started vLLM API server; PID: 640
vLLM API ready: 200
{'object': 'list', 'data': [{'id': 'Qwen/Qwen2.5-1.5B-Instruct', 'object': 'model', 'created': 1793184836, 'owned_by': 'vllm', 'root': 'Qwen/Qwen2.5-1.5B-Instruct', 'parent': None, 'max_model_len': 2048, 'permission': [{'id': 'modelperm-9350e30260596757', 'object': 'model_permission', 'created': 1793184836, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [34]:
import requests

response = requests.post(
    "http://127.0.0.1:8001/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "messages": [
            {
                "role": "user",
                "content": "Briefly explain tenant isolation in LLM inference."
            }
        ],
        "max_tokens": 80,
        "temperature": 0,
    },
    timeout=120,
)

print("HTTP status:", response.status_code)
print(response.text[:3000])
response.raise_for_status()

HTTP status: 200
{"id":"chatcmpl-9f70d16a5bb4076c","object":"chat.completion","created":1793184853,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"Tenant isolation in Large Language Model (LLM) inference refers to the ability of an AI system or model to operate independently and securely within its own environment without interfering with other tenants' data or operations. This concept is crucial for maintaining privacy, security, and integrity when using large language models.\n\nHere’s how tenant isolation works:\n\n1. **Separate Environments**: Each tenant's data and operations run","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"length","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.32.0-7585985b","usage":{"prompt_tokens":39,"total_tokens":119,"completion_tokens":80,"prompt_tokens_details":null,"comp

In [35]:
%cd /kaggle/working/conf-kv-1

print("===== ZONE ENUM =====")
!grep -n -A35 -B5 "class Zone" app/models/enums.py

print("\n===== BACKEND REGISTRATION CLI =====")
!sed -n '1,220p' scripts/register_backend.py

print("\n===== TENANT CREATION CLI =====")
!sed -n '1,240p' scripts/create_tenant.py

/kaggle/working/conf-kv-1
===== ZONE ENUM =====
6-class SecurityLevel(str, Enum):
7-    HIGH = "high"
8-    STANDARD = "standard"
9-
10-
11:class Zone(str, Enum):
12-    SECURE = "secure"
13-    GENERAL = "general"
14-
15-
16-class BackendStatus(str, Enum):
17-    AVAILABLE = "available"
18-    UNAVAILABLE = "unavailable"
19-
20-
21-class HealthStatus(str, Enum):
22-    HEALTHY = "healthy"
23-    DEGRADED = "degraded"
24-    UNHEALTHY = "unhealthy"
25-    UNKNOWN = "unknown"
26-
27-
28-class RequestLifecycleState(str, Enum):
29-    RECEIVED = "received"
30-    AUTHENTICATED = "authenticated"
31-    METADATA_ENRICHED = "metadata_enriched"
32-    SCHEDULED = "scheduled"
33-    POLICY_EVALUATED = "policy_evaluated"
34-    ADMITTED = "admitted"
35-    REJECTED = "rejected"
36-    ROUTED = "routed"
37-    COMPLETED = "completed"
38-    FAILED = "failed"
39-
40-
41-class VerifKVEventType(str, Enum):
42-    AUTHZ_DECISION = "authz_decision"
43-    ROUTE_ASSIGNED = "route_assigned"
44-
45-    

In [36]:
%cd /kaggle/working/conf-kv-1

print("===== BACKEND REGISTRATION ARGUMENTS =====")
!/kaggle/working/ckv-venv/bin/python scripts/register_backend.py --help

print("\n===== TENANT CREATION ARGUMENTS =====")
!/kaggle/working/ckv-venv/bin/python scripts/create_tenant.py --help

print("\n===== GATEWAY CONFIGURATION FIELDS =====")
!grep -nE "class Settings|opa_url|verifkv_url|control_plane_db_path|upstream_timeout|hmac|tenant|api_key" app/config/settings.py app/gateway/auth.py

/kaggle/working/conf-kv-1
===== BACKEND REGISTRATION ARGUMENTS =====
Traceback (most recent call last):
  File "/kaggle/working/conf-kv-1/scripts/register_backend.py", line 6, in <module>
    from app.config.settings import get_settings
ModuleNotFoundError: No module named 'app'

===== TENANT CREATION ARGUMENTS =====
Traceback (most recent call last):
  File "/kaggle/working/conf-kv-1/scripts/create_tenant.py", line 6, in <module>
    from app.config.settings import get_settings
ModuleNotFoundError: No module named 'app'

===== GATEWAY CONFIGURATION FIELDS =====
app/config/settings.py:9:class Settings(BaseSettings):
app/config/settings.py:43:    opa_url: str = "http://127.0.0.1:8181"
app/config/settings.py:50:    verifkv_url: str = "http://127.0.0.1:9000"
app/config/settings.py:56:    upstream_timeout_seconds: float = 60.0
app/config/settings.py:62:    control_plane_db_path: str = "data/control_plane.db"
app/config/settings.py:64:    # Compatibility alias used by existing tenant-regist

In [37]:
%cd /kaggle/working/conf-kv-1

!PYTHONPATH=/kaggle/working/conf-kv-1 /kaggle/working/ckv-venv/bin/python scripts/register_backend.py --help

!PYTHONPATH=/kaggle/working/conf-kv-1 /kaggle/working/ckv-venv/bin/python scripts/create_tenant.py --help

/kaggle/working/conf-kv-1
usage: register_backend.py [-h] --id BACKEND_ID [--endpoint ENDPOINT] --zone
                           {secure,general} [--gpu [GPU ...]] [--available]

Register a ConfidentialKV serving backend

options:
  -h, --help            show this help message and exit
  --id BACKEND_ID
  --endpoint ENDPOINT
  --zone {secure,general}
  --gpu [GPU ...]
  --available
usage: create_tenant.py [-h] --id TENANT_ID --security-level {high,standard}
                        --zones {secure,general} [{secure,general} ...]
                        [--quota QUOTA] [--no-kv-isolation]

Create a ConfidentialKV-Scheduler tenant

options:
  -h, --help            show this help message and exit
  --id TENANT_ID
  --security-level {high,standard}
  --zones {secure,general} [{secure,general} ...]
  --quota QUOTA
  --no-kv-isolation


In [38]:
%cd /kaggle/working/conf-kv-1

!PYTHONPATH=/kaggle/working/conf-kv-1 \
 /kaggle/working/ckv-venv/bin/python scripts/register_backend.py \
 --id kaggle-t4-general \
 --endpoint http://127.0.0.1:8001 \
 --zone general \
 --gpu 0 \
 --available

/kaggle/working/conf-kv-1
Registered backend: kaggle-t4-general


In [39]:
%cd /kaggle/working/conf-kv-1

!PYTHONPATH=/kaggle/working/conf-kv-1 \
 /kaggle/working/ckv-venv/bin/python scripts/create_tenant.py \
 --id kaggle-standard-tenant \
 --security-level standard \
 --zones general \
 --quota 100

/kaggle/working/conf-kv-1
Created tenant: kaggle-standard-tenant
API key — save this now; it will not be displayed again:
cks_RsY3ouHg6wOwStUK0C_59TFXK6UTtamgvLrAWC8tPcQ


In [40]:
%cd /kaggle/working/conf-kv-1

!sed -n '1,120p' app/config/settings.py
!sed -n '1,200p' app/main.py
!sed -n '1,130p' app/gateway/auth.py

/kaggle/working/conf-kv-1
from __future__ import annotations

from functools import lru_cache

from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """
    Centralized ConfidentialKV-Scheduler configuration.

    Configuration can come from environment variables or a local
    .env file.

    The control plane uses centralized configuration so that:
        - OPA endpoint
        - VerifKV endpoint
        - serving backend timeout
        - database paths
        - backend configuration
        - health monitoring
        - VerifKV reconciliation

    are all controlled from one place.
    """

    model_config = SettingsConfigDict(
        env_file=".env",
        extra="ignore",
    )

    # ------------------------------------------------------------------
    # Application
    # ------------------------------------------------------------------

    app_name: str = "ConfidentialKV-Scheduler Control Plane"

In [43]:
%cd /kaggle/working/conf-kv-1

!grep -nE "VerifKVStore|VerifKVClient|hmac_secret|db_path|health_monitor" app/main.py
!grep -nE "def __init__|hmac_secret|db_path|/v1/events|/v1/verify" app/verifkv/client.py app/verifkv/store.py

/kaggle/working/conf-kv-1
33:from app.registry.health_monitor import BackendHealthMonitor
40:from app.verifkv.store import VerifKVStore
120:    tenant_registry = TenantRegistry(settings.tenant_db_path)
122:    backend_registry = BackendRegistry(settings.control_plane_db_path)
141:    verifkv_store = VerifKVStore(
142:        db_path=settings.verifkv_db_path,
143:        hmac_secret=settings.verifkv_hmac_secret.get_secret_value(),
155:    health_monitor = BackendHealthMonitor(
198:        await health_monitor.start()
201:        await health_monitor.check_all()
240:        app.state.health_monitor = health_monitor
258:        await health_monitor.stop()
393:    verifkv_store: VerifKVStore = request.app.state.verifkv_store
app/verifkv/client.py:24:    def __init__(
app/verifkv/client.py:150:            "/v1/events",
app/verifkv/store.py:100:    def __init__(
app/verifkv/store.py:102:        db_path: str,
app/verifkv/store.py:103:        hmac_secret: str,
app/verifkv/store.py:105:        

In [44]:
%cd /kaggle/working/conf-kv-1

from pathlib import Path

sections = {
    "app/main.py": [(115, 175), (185, 210)],
    "app/verifkv/client.py": [(20, 75), (135, 175)],
    "app/config/settings.py": [(40, 90)],
}

for filename, ranges in sections.items():
    lines = Path(filename).read_text().splitlines()
    print(f"\n--- {filename} ---")
    for start, end in ranges:
        print(f"\nLines {start}-{end}:")
        for i in range(start, min(end, len(lines)) + 1):
            print(f"{i}: {lines[i-1]}")

/kaggle/working/conf-kv-1

--- app/main.py ---

Lines 115-175:
115: async def lifespan(app: FastAPI):
116:     """
117:     Initialize all control-plane dependencies before accepting traffic.
118:     """
119: 
120:     tenant_registry = TenantRegistry(settings.tenant_db_path)
121: 
122:     backend_registry = BackendRegistry(settings.control_plane_db_path)
123: 
124:     opa_client = OPAClient(
125:         base_url=settings.opa_url,
126:         timeout_seconds=settings.opa_timeout_seconds,
127:     )
128: 
129:     # ------------------------------------------------------------------
130:     # VerifKV asynchronous event client
131:     # ------------------------------------------------------------------
132: 
133:     verifkv_client = VerifKVEventClient(
134:         base_url=settings.verifkv_url,
135:     )
136: 
137:     # ------------------------------------------------------------------
138:     # VerifKV durable authorization outbox + ordered writer
139:     # -----------------

In [45]:
%cd /kaggle/working/conf-kv-1

import os
from pathlib import Path

assert os.environ.get("VERIFKV_HMAC_SECRET"), (
    "VERIFKV_HMAC_SECRET is missing. Restore the existing secret "
    "used by the VerifKV service before continuing."
)

os.environ["OPA_URL"] = "http://127.0.0.1:8181"
os.environ["VERIFKV_URL"] = "http://127.0.0.1:8100"
os.environ["CONTROL_PLANE_DB_PATH"] = str(
    Path("/kaggle/working/conf-kv-1/data/control_plane.db")
)
os.environ["VERIFKV_DB_PATH"] = str(
    Path("/kaggle/working/conf-kv-1/data/verifkv.db")
)

print("Gateway environment configured.")
print("OPA:", os.environ["OPA_URL"])
print("External VerifKV:", os.environ["VERIFKV_URL"])
print("HMAC secret present:", bool(os.environ.get("VERIFKV_HMAC_SECRET")))

/kaggle/working/conf-kv-1
Gateway environment configured.
OPA: http://127.0.0.1:8181
External VerifKV: http://127.0.0.1:8100
HMAC secret present: True


In [46]:
%cd /kaggle/working/conf-kv-1

!tail -n 35 app/main.py
!grep -nE '^(app|application)[[:space:]]*=|uvicorn.run|def create_app|async def lifespan' app/main.py

/kaggle/working/conf-kv-1
        ) from exc

    except Exception as exc:
        metrics_registry.backend_error_count += 1
        metrics_registry.request_error_count += 1

        logger.exception(
            "Unexpected enforcement failure " "tenant=%s backend=%s",
            tenant.tenant_id,
            approved_route.backend_id,
        )

        raise HTTPException(
            status_code=500,
            detail="Inference request failed",
        ) from exc

    # ==============================================================
    # 10. Return backend response
    # ==============================================================

    content_type = upstream_response.headers.get(
        "content-type",
        "application/json",
    )

    return Response(
        content=upstream_response.content,
        status_code=upstream_response.status_code,
        media_type=content_type,
        headers={
            "X-Backend-ID": approved_route.backend_id,
            "X-Reque

In [47]:
%cd /kaggle/working/conf-kv-1

import os
import subprocess
from pathlib import Path

project = Path("/kaggle/working/conf-kv-1")
venv_python = "/kaggle/working/ckv-venv/bin/python"

gateway_env = os.environ.copy()
gateway_env["PYTHONPATH"] = str(project)
gateway_env["OPA_URL"] = "http://127.0.0.1:8181"
gateway_env["VERIFKV_URL"] = "http://127.0.0.1:8100"
gateway_env["CONTROL_PLANE_DB_PATH"] = str(project / "data/control_plane.db")
gateway_env["VERIFKV_DB_PATH"] = str(project / "data/verifkv.db")

assert gateway_env.get("VERIFKV_HMAC_SECRET"), "Missing HMAC secret"

gateway_log = open("/kaggle/working/gateway.log", "w")

gateway_process = subprocess.Popen(
    [
        venv_python, "-m", "uvicorn",
        "app.main:app",
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    cwd=str(project),
    env=gateway_env,
    stdout=gateway_log,
    stderr=subprocess.STDOUT,
)

print("Gateway process started; PID:", gateway_process.pid)

/kaggle/working/conf-kv-1
Gateway process started; PID: 848


In [48]:
import time
import requests

time.sleep(3)

try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print("HTTP status:", response.status_code)
    print("Response:", response.text[:1000])
except Exception as exc:
    print("Gateway health request failed:", type(exc).__name__)

if gateway_process.poll() is not None:
    print("\nGateway exited. Recent logs:")
    print(Path("/kaggle/working/gateway.log").read_text()[-5000:])

HTTP status: 200
Response: {"status":"ok","service":"ConfidentialKV-Scheduler Control Plane"}


In [51]:
import getpass
import requests

tenant_api_key = getpass.getpass("Enter the tenant API key: ")

Enter the tenant API key:  ········


In [52]:
import json
import requests

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {"role": "user", "content": "Reply with exactly: ConfidentialKV test successful"}
    ],
    "max_tokens": 40,
    "temperature": 0
}

response = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {tenant_api_key}",
        "Content-Type": "application/json",
    },
    json=payload,
    timeout=90,
)

print("HTTP status:", response.status_code)
print("Request ID:", response.headers.get("X-Request-ID"))
print("Backend ID:", response.headers.get("X-Backend-ID"))

try:
    print(json.dumps(response.json(), indent=2)[:2500])
except ValueError:
    print(response.text[:1500])

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))

In [53]:
from pathlib import Path
import requests

print("Gateway process status:")
try:
    print("Poll:", gateway_process.poll())
except NameError:
    print("gateway_process variable is missing.")

print("\nGateway log (last 5000 characters):")
log_path = Path("/kaggle/working/gateway.log")
print(log_path.read_text()[-5000:] if log_path.exists() else "Log file not found.")

print("\nService checks:")
for name, url in [
    ("Gateway", "http://127.0.0.1:8000/health"),
    ("OPA", "http://127.0.0.1:8181/health"),
    ("VerifKV", "http://127.0.0.1:8100/health"),
    ("vLLM", "http://127.0.0.1:8001/v1/models"),
]:
    try:
        r = requests.get(url, timeout=4)
        print(f"{name}: HTTP {r.status_code}")
    except requests.RequestException as e:
        print(f"{name}: UNREACHABLE ({type(e).__name__})")

Gateway process status:
Poll: 0

Gateway log (last 5000 characters):
INFO:     Started server process [848]
INFO:     Waiting for application startup.
2026-10-28 11:08:57,081 | INFO | request_id=- | app.main | Initializing ConfidentialKV-Scheduler control plane
2026-10-28 11:08:57,141 | INFO | request_id=- | app.verifkv.client | VerifKV client started: base_url=http://127.0.0.1:8100
2026-10-28 11:08:57,230 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
2026-10-28 11:08:57,231 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
2026-10-28 11:08:57,231 | INFO | request_id=- | app.main | ConfidentialKV-Scheduler control plane initialized
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
2026-10-28 11:09:02,235 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
2026-10-28 11:09:07,237 | I

In [54]:
import os
import subprocess
import requests
import time
from pathlib import Path

gateway_url = "http://127.0.0.1:8000"
project = Path("/kaggle/working/conf-kv-1")

try:
    r = requests.get(f"{gateway_url}/health", timeout=2)
    print("Gateway is already reachable:", r.status_code)
except requests.RequestException:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(project)
    env["OPA_URL"] = "http://127.0.0.1:8181"
    env["VERIFKV_URL"] = "http://127.0.0.1:8100"
    env["CONTROL_PLANE_DB_PATH"] = str(project / "data/control_plane.db")
    env["VERIFKV_DB_PATH"] = str(project / "data/verifkv.db")

    assert env.get("VERIFKV_HMAC_SECRET"), (
        "HMAC secret missing from this kernel. Do not generate a new one yet."
    )

    gateway_log = open("/kaggle/working/gateway.log", "a")
    gateway_process = subprocess.Popen(
        [
            "/kaggle/working/ckv-venv/bin/python",
            "-m", "uvicorn", "app.main:app",
            "--host", "127.0.0.1", "--port", "8000",
        ],
        cwd=str(project),
        env=env,
        stdout=gateway_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    time.sleep(3)
    print("Gateway process status:", gateway_process.poll())

    try:
        r = requests.get(f"{gateway_url}/health", timeout=5)
        print("Gateway health:", r.status_code, r.text[:500])
    except requests.RequestException as e:
        print("Gateway still unreachable:", type(e).__name__)
        print(Path("/kaggle/working/gateway.log").read_text()[-3000:])

Gateway process status: None
Gateway health: 200 {"status":"ok","service":"ConfidentialKV-Scheduler Control Plane"}


In [55]:
import requests
import time

time.sleep(5)

try:
    r = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print("Gateway after 5 seconds:", r.status_code)
except requests.RequestException as e:
    print("Gateway unreachable:", type(e).__name__)

Gateway after 5 seconds: 200


In [56]:
import requests
import json

assert "tenant_api_key" in globals(), (
    "API key isn't available in this kernel. Re-enter it privately first."
)

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: ConfidentialKV test successful"
        }
    ],
    "max_tokens": 40,
    "temperature": 0
}

try:
    response = requests.post(
        "http://127.0.0.1:8000/v1/chat/completions",
        headers={"Authorization": f"Bearer {tenant_api_key}"},
        json=payload,
        timeout=90,
    )

    print("HTTP status:", response.status_code)
    print("Request ID:", response.headers.get("X-Request-ID"))
    print("Backend ID:", response.headers.get("X-Backend-ID"))

    try:
        print(json.dumps(response.json(), indent=2)[:2000])
    except ValueError:
        print(response.text[:1000])

except requests.RequestException as e:
    print("Request failed:", type(e).__name__, str(e)[:500])

HTTP status: 503
Request ID: 91c99449-73cb-41b0-807a-9592e7165854
Backend ID: None
{
  "detail": "No eligible backend is currently available"
}


In [57]:
import sqlite3
from pathlib import Path
import requests

db_path = "/kaggle/working/conf-kv-1/data/control_plane.db"

print("=== 1. BACKEND REGISTRY ===")
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row

tables = [
    r["name"] for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )
]
print("Tables:", tables)

for table in tables:
    if "backend" in table.lower():
        print(f"\nTable: {table}")
        cols = con.execute(f'PRAGMA table_info("{table}")').fetchall()
        print("Columns:", [c["name"] for c in cols])
        rows = con.execute(f'SELECT * FROM "{table}" LIMIT 10').fetchall()
        for row in rows:
            print(dict(row))

con.close()

print("\n=== 2. SERVICE HEALTH ===")
for name, url in [
    ("vLLM", "http://127.0.0.1:8001/health"),
    ("OPA", "http://127.0.0.1:8181/health"),
    ("VerifKV", "http://127.0.0.1:8100/health"),
]:
    try:
        r = requests.get(url, timeout=4)
        print(f"{name}: HTTP {r.status_code} — {r.text[:200]}")
    except requests.RequestException as e:
        print(f"{name}: {type(e).__name__}")

print("\n=== 3. RECENT GATEWAY LOGS ===")
log_path = Path("/kaggle/working/gateway.log")
if log_path.exists():
    lines = log_path.read_text(errors="replace").splitlines()
    keywords = ("eligible", "backend", "scheduler", "health", "503", "ERROR", "WARNING")
    matches = [line for line in lines if any(k.lower() in line.lower() for k in keywords)]
    print("\n".join(matches[-35:]) or "No matching log lines.")
else:
    print("Gateway log not found.")

=== 1. BACKEND REGISTRY ===
Tables: ['backends', 'tenants', 'credentials']

Table: backends
Columns: ['backend_id', 'endpoint', 'zone', 'gpu_ids', 'configured_status', 'capacity_available', 'load_percent', 'created_at', 'updated_at']
{'backend_id': 'kaggle-t4-general', 'endpoint': 'http://127.0.0.1:8001/', 'zone': 'general', 'gpu_ids': '[0]', 'configured_status': 'available', 'capacity_available': 1, 'load_percent': None, 'created_at': '2026-10-28T11:00:19.342064+00:00', 'updated_at': '2026-10-28T11:00:19.342064+00:00'}

=== 2. SERVICE HEALTH ===
vLLM: ConnectionError
OPA: ConnectionError
VerifKV: ConnectionError

=== 3. RECENT GATEWAY LOGS ===
2026-10-28 11:08:57,081 | INFO | request_id=- | app.main | Initializing ConfidentialKV-Scheduler control plane
2026-10-28 11:08:57,230 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
2026-10-28 11:08:57,231 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 20

In [58]:
import requests
import subprocess

services = {
    "Gateway": "http://127.0.0.1:8000/health",
    "vLLM": "http://127.0.0.1:8001/health",
    "OPA": "http://127.0.0.1:8181/health",
    "VerifKV": "http://127.0.0.1:8100/health",
}

print("=== HTTP checks ===")
for name, url in services.items():
    try:
        r = requests.get(url, timeout=3)
        print(f"{name}: HTTP {r.status_code}")
    except requests.RequestException:
        print(f"{name}: UNREACHABLE")

print("\n=== Saved process handles ===")
for name in ["gateway_process", "vllm_api_process", "opa_process", "verifkv_process"]:
    proc = globals().get(name)
    if proc is None:
        print(f"{name}: no handle in this notebook kernel")
    else:
        print(f"{name}: {'running' if proc.poll() is None else 'exited'}")

=== HTTP checks ===
Gateway: HTTP 200
vLLM: UNREACHABLE
OPA: UNREACHABLE
VerifKV: UNREACHABLE

=== Saved process handles ===
gateway_process: running
vllm_api_process: exited
opa_process: exited
verifkv_process: exited


In [59]:
import subprocess
import requests
import time
from pathlib import Path

opa_log = open("/kaggle/working/opa.log", "a")

opa_process = subprocess.Popen(
    [
        "/kaggle/working/opa",
        "run",
        "--server",
        "--addr=127.0.0.1:8181",
        "/kaggle/working/conf-kv-1/policies/rego",
    ],
    stdout=opa_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

for _ in range(10):
    try:
        r = requests.get("http://127.0.0.1:8181/health", timeout=2)
        if r.status_code == 200:
            print("OPA is back: HTTP", r.status_code)
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print("OPA did not become healthy. Recent log:")
    print(Path("/kaggle/working/opa.log").read_text()[-3000:])

OPA is back: HTTP 200


In [60]:
import os
import subprocess
import requests
import time
from pathlib import Path

project = Path("/kaggle/working/conf-kv-1")

assert os.environ.get("VERIFKV_HMAC_SECRET"), (
    "The existing HMAC secret is missing. Stop here; don't generate a new one."
)

verifkv_env = os.environ.copy()
verifkv_env["VERIFKV_DB_PATH"] = str(project / "data/verifkv.db")

verifkv_log = open("/kaggle/working/verifkv.log", "a")

verifkv_process = subprocess.Popen(
    [
        "/kaggle/working/ckv-venv/bin/python",
        "-c",
        (
            "import os, uvicorn; "
            "from verifkv_service.app import create_app; "
            "uvicorn.run(create_app("
            "db_path=os.environ['VERIFKV_DB_PATH'], "
            "hmac_secret=os.environ['VERIFKV_HMAC_SECRET']), "
            "host='127.0.0.1', port=8100)"
        ),
    ],
    cwd=str(project),
    env=verifkv_env,
    stdout=verifkv_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

for _ in range(15):
    try:
        r = requests.get("http://127.0.0.1:8100/health", timeout=2)
        if r.status_code == 200:
            print("VerifKV is back: HTTP", r.status_code, r.text[:200])
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print("VerifKV did not become healthy. Recent log:")
    print(Path("/kaggle/working/verifkv.log").read_text()[-3000:])

VerifKV is back: HTTP 200 {"status":"ok","service":"verifkv"}


In [61]:
import subprocess
import requests
import time
from pathlib import Path

vllm_log = open("/kaggle/working/vllm.log", "a")

vllm_api_process = subprocess.Popen(
    [
        "/kaggle/working/ckv-venv/bin/python",
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--host", "127.0.0.1",
        "--port", "8001",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.70",
        "--tensor-parallel-size", "1",
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("vLLM process started. Waiting for model loading...")

for attempt in range(60):
    if vllm_api_process.poll() is not None:
        print("vLLM exited during startup.")
        break

    try:
        r = requests.get("http://127.0.0.1:8001/v1/models", timeout=3)
        if r.status_code == 200:
            print("vLLM is ready: HTTP 200")
            print(r.json())
            break
    except requests.RequestException:
        pass

    time.sleep(5)
else:
    print("vLLM did not become ready within five minutes.")

if vllm_api_process.poll() is not None:
    print("\nRecent vLLM logs:")
    print(Path("/kaggle/working/vllm.log").read_text()[-5000:])

vLLM process started. Waiting for model loading...
vLLM is ready: HTTP 200
{'object': 'list', 'data': [{'id': 'Qwen/Qwen2.5-1.5B-Instruct', 'object': 'model', 'created': 1793186355, 'owned_by': 'vllm', 'root': 'Qwen/Qwen2.5-1.5B-Instruct', 'parent': None, 'max_model_len': 2048, 'permission': [{'id': 'modelperm-9e39eca957eebcd6', 'object': 'model_permission', 'created': 1793186355, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [62]:
import requests

for name, url in [
    ("Gateway", "http://127.0.0.1:8000/health"),
    ("OPA", "http://127.0.0.1:8181/health"),
    ("VerifKV", "http://127.0.0.1:8100/health"),
    ("vLLM", "http://127.0.0.1:8001/v1/models"),
]:
    try:
        r = requests.get(url, timeout=5)
        print(f"{name}: HTTP {r.status_code}")
    except requests.RequestException as e:
        print(f"{name}: UNREACHABLE ({type(e).__name__})")

Gateway: HTTP 200
OPA: HTTP 200
VerifKV: HTTP 200
vLLM: HTTP 200


In [63]:
import requests
import json

assert "tenant_api_key" in globals(), (
    "API key isn't available. Re-enter it privately using getpass."
)

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: ConfidentialKV test successful"
        }
    ],
    "max_tokens": 40,
    "temperature": 0
}

try:
    response = requests.post(
        "http://127.0.0.1:8000/v1/chat/completions",
        headers={"Authorization": f"Bearer {tenant_api_key}"},
        json=payload,
        timeout=90,
    )

    print("HTTP status:", response.status_code)
    print("Request ID:", response.headers.get("X-Request-ID"))
    print("Backend ID:", response.headers.get("X-Backend-ID"))

    try:
        print(json.dumps(response.json(), indent=2)[:2000])
    except ValueError:
        print(response.text[:1000])

except requests.RequestException as e:
    print("Request failed:", type(e).__name__, str(e)[:300])

HTTP status: 200
Request ID: 409eb3f6-2406-4793-a7be-89b41575ffeb
Backend ID: kaggle-t4-general
data: {"id":"chatcmpl-94ea4211fbb47b5c","object":"chat.completion.chunk","created":1793186400,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"delta":{"role":"assistant","content":""},"logprobs":null,"finish_reason":null}],"prompt_token_ids":null,"prompt_text":null}

data: {"id":"chatcmpl-94ea4211fbb47b5c","object":"chat.completion.chunk","created":1793186400,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"delta":{"content":"Conf"},"logprobs":null,"finish_reason":null,"token_ids":null}]}

data: {"id":"chatcmpl-94ea4211fbb47b5c","object":"chat.completion.chunk","created":1793186400,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"delta":{"content":"idential"},"logprobs":null,"finish_reason":null,"token_ids":null}]}

data: {"id":"chatcmpl-94ea4211fbb47b5c","object":"chat.completion.chunk","created":1793186400,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"i

In [64]:
import requests

url = "http://127.0.0.1:8100/v1/verify"

try:
    r = requests.get(url, timeout=10)
    print("HTTP status:", r.status_code)
    print("Response:", r.text[:2500])
except requests.RequestException as e:
    print("Verification request failed:", type(e).__name__)

HTTP status: 200
Response: {"valid":false,"failed_sequence":1,"reason":"chain_link_mismatch","verified_events":0}


In [65]:
import requests
import json

r = requests.get("http://127.0.0.1:8100/v1/verify", timeout=10)

print("HTTP status:", r.status_code)

try:
    result = r.json()
    print(json.dumps(result, indent=2)[:3000])
except ValueError:
    print(r.text[:2000])

HTTP status: 200
{
  "valid": false,
  "failed_sequence": 1,
  "reason": "chain_link_mismatch",
  "verified_events": 0
}


In [67]:
%cd /kaggle/working/conf-kv-1

from pathlib import Path
import sqlite3

print("=== VERIFIER LOGIC ===")
p = Path("verifkv_service/store.py")
lines = p.read_text().splitlines()

terms = ("chain_link_mismatch", "previous_hash", "prev_hash", "genesis")
hits = [i for i, line in enumerate(lines) if any(t in line.lower() for t in terms)]

for i in hits[:4]:
    start, end = max(0, i-2), min(len(lines), i+3)
    print(f"\nLines {start+1}-{end}:")
    for j in range(start, end):
        print(f"{j+1}: {lines[j]}")

print("\n=== VERIFKV DATABASE SCHEMA ===")
db = Path("data/verifkv.db")
if db.exists():
    con = sqlite3.connect(str(db))
    tables = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()
    for (table,) in tables:
        cols = con.execute(f'PRAGMA table_info("{table}")').fetchall()
        print(table, "=>", [c[1] for c in cols])
    con.close()
else:
    print("Database not found.")

/kaggle/working/conf-kv-1
=== VERIFIER LOGIC ===

Lines 23-27:
23: 
24:         event_hash =
25:             SHA256(previous_hash + canonical_json)
26: 
27:     Authentication:

Lines 102-106:
102:                     event_timestamp TEXT NOT NULL,
103: 
104:                     previous_hash TEXT NOT NULL,
105: 
106:                     event_hash TEXT NOT NULL,

Lines 136-140:
136:     # ---------------------------------------------------------
137: 
138:     def _get_previous_hash(
139:         self,
140:         connection: sqlite3.Connection,

Lines 149-153:
149: 
150:         if row is None:
151:             return "GENESIS"
152: 
153:         return str(row["event_hash"])

=== VERIFKV DATABASE SCHEMA ===
verifkv_events => ['sequence_number', 'event_id', 'event_type', 'source', 'tenant_id', 'request_id', 'payload_json', 'event_timestamp', 'previous_hash', 'event_hash', 'hmac_tag', 'stored_at']
sqlite_sequence => ['name', 'seq']
authz_outbox => ['outbox_seq', 'authorization_event_

In [68]:
%cd /kaggle/working/conf-kv-1

import sqlite3
from pathlib import Path

# Inspect the first event's chain metadata only.
con = sqlite3.connect("data/verifkv.db")
con.row_factory = sqlite3.Row

row = con.execute("""
    SELECT sequence_number, event_type, previous_hash, event_hash
    FROM verifkv_events
    ORDER BY sequence_number
    LIMIT 1
""").fetchone()

print("=== FIRST EVENT ===")
print(dict(row) if row else "No events found")

con.close()

# Inspect the verifier and event hash implementation.
p = Path("verifkv_service/store.py")
lines = p.read_text().splitlines()

print("\n=== HASH / VERIFICATION METHODS ===")
for start, end in [(130, 190), (550, 630)]:
    print(f"\nLines {start}-{end}:")
    for i in range(start - 1, min(end, len(lines))):
        line = lines[i]
        if any(term in line.lower() for term in (
            "def ", "previous_hash", "event_hash", "sha256",
            "canonical", "genesis", "valid", "mismatch",
        )):
            print(f"{i+1}: {line}")

/kaggle/working/conf-kv-1
=== FIRST EVENT ===
{'sequence_number': 1, 'event_type': 'AUTHZ_DECISION', 'previous_hash': '0000000000000000000000000000000000000000000000000000000000000000', 'event_hash': '690946ccbad90904c24c8f938b7ba14380688ee1a1d120484d3f480bebe40ec2'}

=== HASH / VERIFICATION METHODS ===

Lines 130-190:
138:     def _get_previous_hash(
144:             SELECT event_hash
151:             return "GENESIS"
153:         return str(row["event_hash"])
156:     # Canonical event representation
160:     def _canonicalize_event(
166:         canonical_payload: dict[str, Any] = {
176:         canonical_json = json.dumps(
177:             canonical_payload,
184:             canonical_json,

Lines 550-630:


In [69]:
%cd /kaggle/working/conf-kv-1

from pathlib import Path

for filename in [
    "verifkv_service/store.py",
    "app/verifkv/store.py",
]:
    p = Path(filename)
    if not p.exists():
        continue

    lines = p.read_text().splitlines()
    print(f"\n--- {filename} ---")

    for i, line in enumerate(lines):
        if any(term in line.lower() for term in (
            "def verify", "def _canonicalize_event",
            "def _get_previous_hash", "chain_link_mismatch",
        )):
            start, end = max(0, i-2), min(len(lines), i+22)
            print(f"\nLines {start+1}-{end}:")
            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")

/kaggle/working/conf-kv-1

--- verifkv_service/store.py ---

Lines 136-159:
136:     # ---------------------------------------------------------
137: 
138:     def _get_previous_hash(
139:         self,
140:         connection: sqlite3.Connection,
141:     ) -> str:
142: 
143:         row = connection.execute("""
144:             SELECT event_hash
145:             FROM verifkv_events
146:             ORDER BY sequence_number DESC
147:             LIMIT 1
148:             """).fetchone()
149: 
150:         if row is None:
151:             return "GENESIS"
152: 
153:         return str(row["event_hash"])
154: 
155:     # ---------------------------------------------------------
156:     # Canonical event representation
157:     # ---------------------------------------------------------
158: 
159:     @staticmethod

Lines 158-181:
158: 
159:     @staticmethod
160:     def _canonicalize_event(
161:         event: VerifKVEvent,
162:     ) -> tuple[str, str]:
163: 
164:         timestamp = 

In [70]:
%cd /kaggle/working/conf-kv-1

from pathlib import Path

p = Path("verifkv_service/store.py")
lines = p.read_text().splitlines()

# Print only the relevant sections.
for start, end in [(180, 215), (275, 300), (315, 350)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

/kaggle/working/conf-kv-1

--- Lines 180-215 ---
180:             ensure_ascii=False,
181:         )
182: 
183:         return (
184:             canonical_json,
185:             timestamp,
186:         )
187: 
188:     # ---------------------------------------------------------
189:     # Append event
190:     # ---------------------------------------------------------
191: 
192:     def append(
193:         self,
194:         event: VerifKVEvent,
195:     ) -> int:
196: 
197:         canonical_json, event_timestamp = self._canonicalize_event(event)
198: 
199:         with self._lock:
200: 
201:             with self._connect() as connection:
202: 
203:                 previous_hash = self._get_previous_hash(connection)
204: 
205:                 hash_input = (previous_hash + canonical_json).encode("utf-8")
206: 
207:                 event_hash = hashlib.sha256(hash_input).hexdigest()
208: 
209:                 hmac_tag = hmac.new(
210:                     self.hmac_secret,
211:      

In [72]:
%cd /kaggle/working/conf-kv-1
!git pull --ff-only

/kaggle/working/conf-kv-1
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 634 bytes | 317.00 KiB/s, done.
From https://github.com/aditya3275/conf-kv-1
   b1fec8e..7ab641f  main       -> origin/main
Updating b1fec8e..7ab641f
Fast-forward
 tests/unit/test_verifkv.py | 4 ++--
 verifkv_service/store.py   | 5 +++--
 2 files changed, 5 insertions(+), 4 deletions(-)


In [73]:
%cd /kaggle/working/conf-kv-1

!grep -n -A 8 -B 3 'def _get_previous_hash\|def verify_chain' verifkv_service/store.py

/kaggle/working/conf-kv-1
136-    # Previous hash
137-    # ---------------------------------------------------------
138-
139:    def _get_previous_hash(
140-        self,
141-        connection: sqlite3.Connection,
142-    ) -> str:
143-
144-        row = connection.execute("""
145-            SELECT event_hash
146-            FROM verifkv_events
147-            ORDER BY sequence_number DESC
--
255-    # Verify complete hash chain
256-    # ---------------------------------------------------------
257-
258:    def verify_chain(self) -> dict[str, Any]:
259-
260-        with self._connect() as connection:
261-
262-            rows = connection.execute("""
263-                SELECT
264-                    sequence_number,
265-                    event_id,
266-                    event_type,


In [74]:
import requests

for name, url in {
    "Gateway": "http://127.0.0.1:8000/health",
    "OPA": "http://127.0.0.1:8181/health",
    "VerifKV": "http://127.0.0.1:8100/health",
    "vLLM": "http://127.0.0.1:8001/v1/models",
}.items():
    try:
        r = requests.get(url, timeout=3)
        print(f"{name}: HTTP {r.status_code}")
    except requests.RequestException:
        print(f"{name}: NOT REACHABLE")

Gateway: HTTP 200
OPA: HTTP 200
VerifKV: HTTP 200
vLLM: HTTP 200


In [75]:
%cd /kaggle/working/conf-kv-1

!grep -n -A 35 -B 5 'def __init__\|def append\|def verify_chain\|def _get_previous_hash' verifkv_service/store.py

/kaggle/working/conf-kv-1
34-        The exact canonical JSON used to generate the hash is stored
35-        in payload_json and reused during verification. This avoids
36-        re-serialization differences during verification.
37-    """
38-
39:    def __init__(
40-        self,
41-        db_path: str,
42-        hmac_secret: str,
43-    ) -> None:
44-
45-        self.db_path = Path(db_path)
46-
47-        self.db_path.parent.mkdir(
48-            parents=True,
49-            exist_ok=True,
50-        )
51-
52-        self.hmac_secret = hmac_secret.encode("utf-8")
53-
54-        self._lock = Lock()
55-
56-        self._initialize_database()
57-
58-    # ---------------------------------------------------------
59-    # Database connection
60-    # ---------------------------------------------------------
61-
62-    def _connect(self) -> sqlite3.Connection:
63-
64-        connection = sqlite3.connect(
65-            self.db_path,
66-            timeout=10.0,
67-        )
68-
69-    

In [78]:
%cd /kaggle/working/conf-kv-1

!grep -n -E 'def __init__|def append|def verify_chain|def _get_previous_hash|GENESIS_HASH|previous_hash = ' verifkv_service/store.py

/kaggle/working/conf-kv-1
8:from app.models.verifkv import GENESIS_HASH
39:    def __init__(
139:    def _get_previous_hash(
152:            return GENESIS_HASH
193:    def append(
204:                previous_hash = self._get_previous_hash(connection)
258:    def verify_chain(self) -> dict[str, Any]:
279:        previous_hash = GENESIS_HASH
361:            previous_hash = row["event_hash"]


In [79]:
%cd /kaggle/working/conf-kv-1

with open("verifkv_service/store.py") as f:
    lines = f.readlines()

for start, end in [(35, 75), (185, 245), (250, 310)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}", end="")

/kaggle/working/conf-kv-1

--- Lines 35-75 ---
35:         in payload_json and reused during verification. This avoids
36:         re-serialization differences during verification.
37:     """
38: 
39:     def __init__(
40:         self,
41:         db_path: str,
42:         hmac_secret: str,
43:     ) -> None:
44: 
45:         self.db_path = Path(db_path)
46: 
47:         self.db_path.parent.mkdir(
48:             parents=True,
49:             exist_ok=True,
50:         )
51: 
52:         self.hmac_secret = hmac_secret.encode("utf-8")
53: 
54:         self._lock = Lock()
55: 
56:         self._initialize_database()
57: 
58:     # ---------------------------------------------------------
59:     # Database connection
60:     # ---------------------------------------------------------
61: 
62:     def _connect(self) -> sqlite3.Connection:
63: 
64:         connection = sqlite3.connect(
65:             self.db_path,
66:             timeout=10.0,
67:         )
68: 
69:         connection.r

In [80]:
%cd /kaggle/working/conf-kv-1

import tempfile
from pathlib import Path

from app.models.verifkv import GENESIS_HASH, VerifKVEvent
from verifkv_service.store import VerifKVStore

test_dir = tempfile.TemporaryDirectory(prefix="verifkv_test_")
test_db = str(Path(test_dir.name) / "fresh_test.db")

# Use a test-only secret; do not print it.
test_store = VerifKVStore(
    db_path=test_db,
    hmac_secret="temporary-test-secret-only",
)

print("Empty chain:", test_store.verify_chain())

/kaggle/working/conf-kv-1
Empty chain: {'valid': True, 'verified_events': 0}


In [81]:
import inspect
from app.models.verifkv import VerifKVEvent

print(inspect.signature(VerifKVEvent))

(*, event_id: uuid.UUID = <factory>, event_type: app.models.verifkv.VerifKVEventType, source: str, tenant_id: str | None = None, request_id: str | None = None, payload: dict[str, typing.Any] = <factory>, timestamp: datetime.datetime = <factory>) -> None


In [82]:
from app.models.verifkv import VerifKVEvent, VerifKVEventType, GENESIS_HASH

event = VerifKVEvent(
    event_type=next(iter(VerifKVEventType)),
    source="kaggle-fresh-chain-test",
    tenant_id="test-tenant",
    request_id="test-request-001",
    payload={"test": True, "purpose": "genesis-hash-validation"},
)

sequence = test_store.append(event)
result = test_store.verify_chain()

print("Sequence:", sequence)
print("Expected genesis:", GENESIS_HASH)
print("Verification:", result)

Sequence: 1
Expected genesis: 0000000000000000000000000000000000000000000000000000000000000000
Verification: {'valid': True, 'verified_events': 1}


In [83]:
import requests

url = "http://127.0.0.1:8100/v1/verify"

try:
    r = requests.get(url, timeout=10)
    print("HTTP:", r.status_code)
    print("Result:", r.json())
except Exception as e:
    print(type(e).__name__, str(e))

HTTP: 200
Result: {'valid': False, 'failed_sequence': 1, 'reason': 'chain_link_mismatch', 'verified_events': 0}


In [84]:
r = requests.get("http://127.0.0.1:8100/v1/verify", timeout=10)

print("HTTP:", r.status_code)
print("Verification:", r.json())

HTTP: 200
Verification: {'valid': False, 'failed_sequence': 1, 'reason': 'chain_link_mismatch', 'verified_events': 0}


In [85]:
import requests

r = requests.get(
    "http://127.0.0.1:8100/v1/verify",
    timeout=10
)

print(r.json())

{'valid': False, 'failed_sequence': 1, 'reason': 'chain_link_mismatch', 'verified_events': 0}


In [86]:
%cd /kaggle/working/conf-kv-1

!grep -n -E 'GENESIS_HASH|return "GENESIS"|previous_hash = ' verifkv_service/store.py

/kaggle/working/conf-kv-1
8:from app.models.verifkv import GENESIS_HASH
152:            return GENESIS_HASH
204:                previous_hash = self._get_previous_hash(connection)
279:        previous_hash = GENESIS_HASH
361:            previous_hash = row["event_hash"]


In [87]:
import sqlite3

db_path = "/kaggle/working/conf-kv-1/data/verifkv.db"

conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

row = conn.execute("""
    SELECT sequence_number, previous_hash, event_hash
    FROM verifkv_events
    ORDER BY sequence_number ASC
    LIMIT 1
""").fetchone()

print("First event:", row)
print("Expected genesis:", GENESIS_HASH)

conn.close()

First event: (1, '0000000000000000000000000000000000000000000000000000000000000000', '690946ccbad90904c24c8f938b7ba14380688ee1a1d120484d3f480bebe40ec2')
Expected genesis: 0000000000000000000000000000000000000000000000000000000000000000


In [88]:
from app.models.verifkv import GENESIS_HASH

In [89]:
import sqlite3

db_path = "/kaggle/working/conf-kv-1/data/verifkv.db"

conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

row = conn.execute("""
    SELECT sequence_number, previous_hash, event_hash
    FROM verifkv_events
    ORDER BY sequence_number ASC
    LIMIT 1
""").fetchone()

print("First event:", row)
print("Expected genesis:", GENESIS_HASH)

conn.close()

First event: (1, '0000000000000000000000000000000000000000000000000000000000000000', '690946ccbad90904c24c8f938b7ba14380688ee1a1d120484d3f480bebe40ec2')
Expected genesis: 0000000000000000000000000000000000000000000000000000000000000000


In [90]:
import os

print("Configured DB:", os.environ.get("VERIFKV_DB_PATH"))
print("Configured service URL:", os.environ.get("VERIFKV_URL"))

Configured DB: /kaggle/working/conf-kv-1/data/verifkv.db
Configured service URL: http://127.0.0.1:8100


In [91]:
%cd /kaggle/working/conf-kv-1

!sed -n '258,300p' verifkv_service/store.py
!grep -n -E 'VERIFKV_DB_PATH|create_app|VerifKVStore\(' verifkv_service/app.py

/kaggle/working/conf-kv-1
    def verify_chain(self) -> dict[str, Any]:

        with self._connect() as connection:

            rows = connection.execute("""
                SELECT
                    sequence_number,
                    event_id,
                    event_type,
                    source,
                    tenant_id,
                    request_id,
                    payload_json,
                    event_timestamp,
                    previous_hash,
                    event_hash,
                    hmac_tag
                FROM verifkv_events
                ORDER BY sequence_number ASC
                """).fetchall()

        previous_hash = GENESIS_HASH

        verified = 0

        for row in rows:

            # -------------------------------------------------
            # 1. Verify chain linkage
            # -------------------------------------------------

            if row["previous_hash"] != previous_hash:

                return {
             

In [92]:
import requests

r = requests.get(
    "http://127.0.0.1:8100/v1/verify",
    timeout=10
)

print("HTTP:", r.status_code)
print("Result:", r.json())

HTTP: 200
Result: {'valid': False, 'failed_sequence': 1, 'reason': 'chain_link_mismatch', 'verified_events': 0}


In [93]:
%cd /kaggle/working/conf-kv-1

!sed -n '1,65p' verifkv_service/app.py

/kaggle/working/conf-kv-1
from __future__ import annotations

from contextlib import asynccontextmanager

from fastapi import FastAPI, status

from app.models.verifkv import VerifKVEvent

from verifkv_service.store import VerifKVStore
from verifkv_service.watchdog import (
    BackendWatchdog,
)
from verifkv_service.writer import (
    VerifKVWriter,
)


class VerifKVApplicationState:

    def __init__(
        self,
        db_path: str,
        hmac_secret: str,
        queue_maxsize: int,
        watch_interval_seconds: float,
        watch_timeout_seconds: float,
    ) -> None:

        self.store = VerifKVStore(
            db_path=db_path,
            hmac_secret=hmac_secret,
        )

        self.writer = VerifKVWriter(
            store=self.store,
            queue_maxsize=queue_maxsize,
        )

        self.watchdog = BackendWatchdog(
            writer=self.writer,
            interval_seconds=(watch_interval_seconds),
            timeout_seconds=(watch_timeout_seconds)

In [94]:
import os
import requests

print("Notebook DB path:", os.environ.get("VERIFKV_DB_PATH"))
print("Notebook HMAC secret configured:",
      bool(os.environ.get("VERIFKV_HMAC_SECRET")))

!ps -eo pid,args | grep '[u]vicorn'

Notebook DB path: /kaggle/working/conf-kv-1/data/verifkv.db
Notebook HMAC secret configured: True
    858 /kaggle/working/ckv-venv/bin/python -m uvicorn app.main:app --host 127.0.0.1 --port 8000
    875 /kaggle/working/ckv-venv/bin/python -c import os, uvicorn; from verifkv_service.app import create_app; uvicorn.run(create_app(db_path=os.environ['VERIFKV_DB_PATH'], hmac_secret=os.environ['VERIFKV_HMAC_SECRET']), host='127.0.0.1', port=8100)


In [95]:
import os
import sqlite3

db_path = os.path.realpath(os.environ["VERIFKV_DB_PATH"])

print("Resolved DB:", db_path)
print("DB exists:", os.path.exists(db_path))
print("DB size:", os.path.getsize(db_path) if os.path.exists(db_path) else None)

conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

row = conn.execute("""
    SELECT sequence_number, previous_hash, event_hash
    FROM verifkv_events
    ORDER BY sequence_number
    LIMIT 1
""").fetchone()

print("First row:", row)
conn.close()

Resolved DB: /kaggle/working/conf-kv-1/data/verifkv.db
DB exists: True
DB size: 40960
First row: (1, '0000000000000000000000000000000000000000000000000000000000000000', '690946ccbad90904c24c8f938b7ba14380688ee1a1d120484d3f480bebe40ec2')


In [96]:
import os

print("Service working directory:",
      os.readlink("/proc/875/cwd"))
print("Service command line:",
      open("/proc/875/cmdline", "rb").read().replace(b"\0", b" ").decode())

Service working directory: /kaggle/working/conf-kv-1
Service command line: /kaggle/working/ckv-venv/bin/python -c import os, uvicorn; from verifkv_service.app import create_app; uvicorn.run(create_app(db_path=os.environ['VERIFKV_DB_PATH'], hmac_secret=os.environ['VERIFKV_HMAC_SECRET']), host='127.0.0.1', port=8100) 


In [97]:
import requests

# Ask the live service to verify its chain again
r = requests.get("http://127.0.0.1:8100/v1/verify", timeout=10)
print("Live verification:", r.json())

# Show the source code currently on disk around the genesis check
%cd /kaggle/working/conf-kv-1
!sed -n '135,155p' verifkv_service/store.py
!sed -n '275,295p' verifkv_service/store.py

Live verification: {'valid': False, 'failed_sequence': 1, 'reason': 'chain_link_mismatch', 'verified_events': 0}
/kaggle/working/conf-kv-1
    # ---------------------------------------------------------
    # Previous hash
    # ---------------------------------------------------------

    def _get_previous_hash(
        self,
        connection: sqlite3.Connection,
    ) -> str:

        row = connection.execute("""
            SELECT event_hash
            FROM verifkv_events
            ORDER BY sequence_number DESC
            LIMIT 1
            """).fetchone()

        if row is None:
            return GENESIS_HASH

        return str(row["event_hash"])

                FROM verifkv_events
                ORDER BY sequence_number ASC
                """).fetchall()

        previous_hash = GENESIS_HASH

        verified = 0

        for row in rows:

            # -------------------------------------------------
            # 1. Verify chain linkage
            # -------------

In [98]:
import os
import sys

print("Python executable:", os.readlink("/proc/875/exe"))
print("Service PID:", 875)
print("Module file on disk:",
      "/kaggle/working/conf-kv-1/verifkv_service/store.py")

Python executable: /usr/bin/python3.12
Service PID: 875
Module file on disk: /kaggle/working/conf-kv-1/verifkv_service/store.py


In [99]:
import os
import signal
import subprocess
import time
import requests

# Preserve the existing environment, database path, and HMAC secret.
assert os.environ.get("VERIFKV_DB_PATH"), "VERIFKV_DB_PATH is missing"
assert os.environ.get("VERIFKV_HMAC_SECRET"), "HMAC secret is missing"

# Stop only the external VerifKV process.
try:
    os.kill(875, signal.SIGTERM)
except ProcessLookupError:
    pass

time.sleep(2)

# Start VerifKV using the updated source code.
log_file = open("/kaggle/working/verifkv-restart.log", "a")

verifkv_process = subprocess.Popen(
    [
        "/kaggle/working/ckv-venv/bin/python",
        "-c",
        (
            "import os, uvicorn; "
            "from verifkv_service.app import create_app; "
            "uvicorn.run("
            "create_app("
            "db_path=os.environ['VERIFKV_DB_PATH'], "
            "hmac_secret=os.environ['VERIFKV_HMAC_SECRET']"
            "), "
            "host='127.0.0.1', port=8100)"
        ),
    ],
    cwd="/kaggle/working/conf-kv-1",
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("New VerifKV PID:", verifkv_process.pid)

New VerifKV PID: 1084


In [100]:
import requests
import time

for attempt in range(15):
    try:
        health = requests.get(
            "http://127.0.0.1:8100/health", timeout=2
        )
        if health.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)

r = requests.get("http://127.0.0.1:8100/v1/verify", timeout=10)
print("HTTP:", r.status_code)
print("Verification:", r.json())

HTTP: 200
Verification: {'valid': False, 'failed_sequence': 1, 'reason': 'event_hash_mismatch', 'verified_events': 0}


In [101]:
import sqlite3
import hashlib
import json
import os

from app.models.verifkv import GENESIS_HASH

db_path = os.environ["VERIFKV_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row

row = conn.execute("""
    SELECT sequence_number, payload_json, previous_hash, event_hash
    FROM verifkv_events
    ORDER BY sequence_number
    LIMIT 1
""").fetchone()

conn.close()

canonical_json = row["payload_json"]
recomputed_hash = hashlib.sha256(
    (GENESIS_HASH + canonical_json).encode("utf-8")
).hexdigest()

print("Stored previous hash matches genesis:", row["previous_hash"] == GENESIS_HASH)
print("Stored event hash:", row["event_hash"])
print("Recomputed event hash:", recomputed_hash)
print("Hashes match:", row["event_hash"] == recomputed_hash)
print("Stored payload length:", len(canonical_json))

Stored previous hash matches genesis: True
Stored event hash: 690946ccbad90904c24c8f938b7ba14380688ee1a1d120484d3f480bebe40ec2
Recomputed event hash: 5f1d013214e2645d4465d3cec8ad599322d96530877b28752c85d424e5e95c53
Hashes match: False
Stored payload length: 285


In [102]:
import sqlite3
import os

db_path = os.environ["VERIFKV_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row

row = conn.execute("""
    SELECT payload_json, previous_hash, event_hash, hmac_tag
    FROM verifkv_events
    ORDER BY sequence_number
    LIMIT 1
""").fetchone()

print("Stored payload:")
print(row["payload_json"])
print("\nStored previous hash:", row["previous_hash"])
print("Stored event hash:", row["event_hash"])
print("HMAC present:", bool(row["hmac_tag"]))

conn.close()

Stored payload:
{"backend_id":"kaggle-t4-general","decision":true,"decision_id":"3d3500948192ae798f53649b78a7248a8dec38bb3a1c30c6522f1da2a023ee7b","outbox_seq":1,"policy_version":"v1","reason":"allowed","scheduler_reason":"eligible backend selected using zone compatibility and load","zone":"general"}

Stored previous hash: 0000000000000000000000000000000000000000000000000000000000000000
Stored event hash: 690946ccbad90904c24c8f938b7ba14380688ee1a1d120484d3f480bebe40ec2
HMAC present: True


In [103]:
%cd /kaggle/working/conf-kv-1

!sed -n '195,220p' verifkv_service/store.py
!sed -n '300,345p' verifkv_service/store.py
!sed -n '500,545p' app/verifkv/store.py

/kaggle/working/conf-kv-1
        event: VerifKVEvent,
    ) -> int:

        canonical_json, event_timestamp = self._canonicalize_event(event)

        with self._lock:

            with self._connect() as connection:

                previous_hash = self._get_previous_hash(connection)

                hash_input = (previous_hash + canonical_json).encode("utf-8")

                event_hash = hashlib.sha256(hash_input).hexdigest()

                hmac_tag = hmac.new(
                    self.hmac_secret,
                    event_hash.encode("utf-8"),
                    hashlib.sha256,
                ).hexdigest()

                stored_at = datetime.now(timezone.utc).isoformat()

                cursor = connection.execute(
                    """
                    INSERT INTO verifkv_events (
            # -------------------------------------------------

            canonical_json = row["payload_json"]

            # Make sure it is valid JSON.
            try:

            

In [104]:
%cd /kaggle/working/conf-kv-1

!sed -n '520,590p' app/verifkv/store.py

/kaggle/working/conf-kv-1
            """)

        rows = await cursor.fetchall()

        previous_hash = GENESIS_HASH

        for row in rows:
            payload = json.loads(row["payload_json"])

            canonical_event = json.dumps(
                {
                    "event_id": row["event_id"],
                    "event_type": row["event_type"],
                    "source": row["source"],
                    "tenant_id": row["tenant_id"],
                    "request_id": row["request_id"],
                    "payload": payload,
                    "timestamp": row["event_timestamp"],
                    "prev_hash": row["previous_hash"],
                },
                sort_keys=True,
                separators=(",", ":"),
            )

            # ----------------------------------------------------------
            # Previous-hash linkage.
            # ----------------------------------------------------------

            if row["previous_hash"] != previou

In [106]:
%cd /kaggle/working/conf-kv-1

!grep -n -E 'def append|canonical_event|canonical_json|event_hash =|hash_input =' app/verifkv/store.py | head -25

/kaggle/working/conf-kv-1
141:    async def append_authorization_outbox(
376:            canonical_event = json.dumps(
395:            event_hash = hashlib.sha256(
396:                (previous_hash + canonical_event).encode("utf-8")
529:            canonical_event = json.dumps(
559:                (previous_hash + canonical_event).encode("utf-8")


In [107]:
%cd /kaggle/working/conf-kv-1

!sed -n '350,410p' app/verifkv/store.py

/kaggle/working/conf-kv-1
            source = "control-plane"

            # ----------------------------------------------------------
            # Build deterministic event payload.
            # ----------------------------------------------------------

            payload = {
                "decision": record.decision,
                "backend_id": record.backend_id,
                "zone": record.zone,
                "decision_id": record.decision_id,
                "policy_version": record.policy_version,
                "outbox_seq": record.outbox_seq,
                **record.payload,
            }

            # ----------------------------------------------------------
            # Canonical event representation.
            #
            # source participates in the hash because it is part of
            # the persisted VerifKV event identity.
            #
            # stored_at intentionally does NOT participate in hashing;
            # it is persistence metadata 

In [109]:
!grep -n -E 'VerifKVEvent|/v1/events|event_hash|payload_json' app/verifkv/client.py verifkv_service/app.py

app/verifkv/client.py:8:from app.models.verifkv import VerifKVEvent
app/verifkv/client.py:13:class VerifKVEventClient:
app/verifkv/client.py:31:        self.queue: asyncio.Queue[VerifKVEvent] = asyncio.Queue(maxsize=queue_maxsize)
app/verifkv/client.py:89:    def emit_nowait(self, event: VerifKVEvent) -> bool:
app/verifkv/client.py:139:        event: VerifKVEvent,
app/verifkv/client.py:150:            "/v1/events",
verifkv_service/app.py:7:from app.models.verifkv import VerifKVEvent
verifkv_service/app.py:100:        "/v1/events",
verifkv_service/app.py:104:        event: VerifKVEvent,


In [110]:
%cd /kaggle/working/conf-kv-1

!sed -n '130,175p' app/verifkv/client.py
!sed -n '90,120p' verifkv_service/app.py

/kaggle/working/conf-kv-1
                    "Failed to send VerifKV event=%s",
                    event.event_id,
                )

            finally:
                self.queue.task_done()

    async def _send_event(
        self,
        event: VerifKVEvent,
    ) -> None:

        if self._client is None:
            logger.warning(
                "HTTP client unavailable; dropping event=%s",
                event.event_id,
            )
            return

        response = await self._client.post(
            "/v1/events",
            json=event.model_dump(mode="json"),
        )

        response.raise_for_status()

    @app.get("/health")
    async def health() -> dict:

        return {
            "status": "ok",
            "service": "verifkv",
        }

    @app.post(
        "/v1/events",
        status_code=status.HTTP_202_ACCEPTED,
    )
    async def ingest_event(
        event: VerifKVEvent,
    ) -> dict:

        accepted = state.writer.enqueue_nowait(event)

In [114]:
%cd /kaggle/working/conf-kv-1
!sed -n '218,250p' verifkv_service/store.py

/kaggle/working/conf-kv-1
                cursor = connection.execute(
                    """
                    INSERT INTO verifkv_events (
                        event_id,
                        event_type,
                        source,
                        tenant_id,
                        request_id,
                        payload_json,
                        event_timestamp,
                        previous_hash,
                        event_hash,
                        hmac_tag,
                        stored_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        str(event.event_id),
                        event.event_type.value,
                        event.source,
                        event.tenant_id,
                        event.request_id,
                        canonical_json,
                        event_timestamp,
                        previous_has

In [115]:
import requests

services = {
    "vLLM": "http://127.0.0.1:8001/v1/models",
    "Gateway": "http://127.0.0.1:8000/health",
    "OPA": "http://127.0.0.1:8181/health",
    "VerifKV": "http://127.0.0.1:8100/health",
}

for name, url in services.items():
    try:
        response = requests.get(url, timeout=3)
        print(f"{name}: HTTP {response.status_code}")
    except requests.RequestException:
        print(f"{name}: NOT REACHABLE")

vLLM: HTTP 200
Gateway: HTTP 200
OPA: HTTP 200
VerifKV: HTTP 200


In [116]:
import requests
import time
import statistics

VLLM_URL = "http://127.0.0.1:8001/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "Explain cloud computing in exactly 50 words."}
    ],
    "max_tokens": 80,
    "temperature": 0,
}

# Warm-up request; exclude it from measurements.
warmup = requests.post(VLLM_URL, json=payload, timeout=90)
warmup.raise_for_status()
print("Warm-up: HTTP", warmup.status_code)

latencies = []
for i in range(10):
    start = time.perf_counter()
    response = requests.post(VLLM_URL, json=payload, timeout=90)
    elapsed = time.perf_counter() - start
    response.raise_for_status()
    latencies.append(elapsed)

    print(f"Run {i+1:02d}: {elapsed:.3f}s")

print("\n--- Direct vLLM baseline ---")
print(f"Successful requests: {len(latencies)}/10")
print(f"Mean latency:   {statistics.mean(latencies):.3f}s")
print(f"Median latency: {statistics.median(latencies):.3f}s")
print(f"Min latency:    {min(latencies):.3f}s")
print(f"Max latency:    {max(latencies):.3f}s")

Warm-up: HTTP 200
Run 01: 1.794s
Run 02: 1.833s
Run 03: 1.882s
Run 04: 1.939s
Run 05: 1.954s
Run 06: 2.014s
Run 07: 1.984s
Run 08: 1.982s
Run 09: 1.964s
Run 10: 1.902s

--- Direct vLLM baseline ---
Successful requests: 10/10
Mean latency:   1.925s
Median latency: 1.947s
Min latency:    1.794s
Max latency:    2.014s


In [117]:
import requests
import time
import statistics

GATEWAY_URL = "http://127.0.0.1:8000/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {tenant_api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain cloud computing in exactly 50 words."
        }
    ],
    "max_tokens": 80,
    "temperature": 0,
}

# Warm-up: exclude from measurements.
warmup = requests.post(
    GATEWAY_URL, headers=headers, json=payload, timeout=90
)
print("Warm-up HTTP:", warmup.status_code)

if warmup.status_code != 200:
    print(warmup.text[:500])
    raise RuntimeError("Gateway warm-up failed; fix this before benchmarking.")

latencies = []

for i in range(10):
    start = time.perf_counter()
    response = requests.post(
        GATEWAY_URL, headers=headers, json=payload, timeout=90
    )
    elapsed = time.perf_counter() - start

    if response.status_code != 200:
        print(f"Run {i+1:02d}: HTTP {response.status_code}")
        print(response.text[:300])
        continue

    latencies.append(elapsed)
    print(f"Run {i+1:02d}: {elapsed:.3f}s")

if latencies:
    print("\n--- ConfidentialKV gateway results ---")
    print(f"Successful requests: {len(latencies)}/10")
    print(f"Mean latency:   {statistics.mean(latencies):.3f}s")
    print(f"Median latency: {statistics.median(latencies):.3f}s")
    print(f"Min latency:    {min(latencies):.3f}s")
    print(f"Max latency:    {max(latencies):.3f}s")

Warm-up HTTP: 200
Run 01: 1.168s
Run 02: 1.838s
Run 03: 1.091s
Run 04: 2.040s
Run 05: 1.235s
Run 06: 1.376s
Run 07: 2.088s
Run 08: 1.752s
Run 09: 1.743s
Run 10: 1.930s

--- ConfidentialKV gateway results ---
Successful requests: 10/10
Mean latency:   1.626s
Median latency: 1.748s
Min latency:    1.091s
Max latency:    2.088s


In [118]:
import requests
import time
import statistics

DIRECT = "http://127.0.0.1:8001/v1/chat/completions"
GATEWAY = "http://127.0.0.1:8000/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {tenant_api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {"role": "user",
         "content": "Explain cloud computing in exactly 50 words."}
    ],
    "max_tokens": 80,
    "temperature": 0,
}

def timed_request(url, use_auth=False):
    start = time.perf_counter()
    r = requests.post(
        url,
        headers=headers if use_auth else None,
        json=payload,
        timeout=90,
    )
    elapsed = time.perf_counter() - start
    r.raise_for_status()
    return elapsed

# Warm both paths, excluding warm-ups from the data.
timed_request(DIRECT)
timed_request(GATEWAY, use_auth=True)

direct, gateway = [], []

for i in range(20):
    # Alternate request order to reduce order effects.
    order = (
        [(DIRECT, False, direct), (GATEWAY, True, gateway)]
        if i % 2 == 0
        else [(GATEWAY, True, gateway), (DIRECT, False, direct)]
    )

    for url, auth, results in order:
        results.append(timed_request(url, use_auth=auth))

print("Direct mean:", round(statistics.mean(direct), 3), "s")
print("Direct median:", round(statistics.median(direct), 3), "s")
print("Gateway mean:", round(statistics.mean(gateway), 3), "s")
print("Gateway median:", round(statistics.median(gateway), 3), "s")
print("Direct successes:", len(direct), "/ 20")
print("Gateway successes:", len(gateway), "/ 20")

Direct mean: 1.652 s
Direct median: 1.611 s
Gateway mean: 1.501 s
Gateway median: 1.424 s
Direct successes: 20 / 20
Gateway successes: 20 / 20


In [119]:
import requests
import time
import statistics

DIRECT = "http://127.0.0.1:8001/v1/chat/completions"
GATEWAY = "http://127.0.0.1:8000/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {tenant_api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": (
                "Explain cloud computing, its architecture, "
                "deployment models, security challenges, and "
                "practical applications in detail."
            ),
        }
    ],
    "max_tokens": 128,
    "temperature": 0,
}

def measure(url, auth=False):
    start = time.perf_counter()
    r = requests.post(
        url,
        headers=headers if auth else None,
        json=payload,
        timeout=90,
    )
    elapsed = time.perf_counter() - start
    r.raise_for_status()

    data = r.json()
    usage = data.get("usage", {})
    tokens = usage.get("completion_tokens")

    if tokens is None:
        raise RuntimeError(
            "API did not return completion_tokens; "
            "cannot calculate reliable token throughput."
        )

    return elapsed, tokens, tokens / elapsed

# Warm-up both paths.
measure(DIRECT)
measure(GATEWAY, auth=True)

results = {"Direct": [], "Gateway": []}

for i in range(10):
    order = (
        [("Direct", DIRECT, False), ("Gateway", GATEWAY, True)]
        if i % 2 == 0
        else [("Gateway", GATEWAY, True), ("Direct", DIRECT, False)]
    )

    for name, url, auth in order:
        elapsed, tokens, rate = measure(url, auth)
        results[name].append((elapsed, tokens, rate))
        print(
            f"Pair {i+1:02d} | {name}: "
            f"{elapsed:.3f}s, {tokens} tokens, {rate:.2f} tokens/s"
        )

for name, rows in results.items():
    print(f"\n--- {name} ---")
    print("Mean latency:", round(statistics.mean(x[0] for x in rows), 3), "s")
    print("Mean output tokens:", round(statistics.mean(x[1] for x in rows), 1))
    print("Mean per-request token rate:", round(statistics.mean(x[2] for x in rows), 2), "tokens/s")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [120]:
import requests

url = "http://127.0.0.1:8000/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {tenant_api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain cloud computing in detail."
        }
    ],
    "max_tokens": 128,
    "temperature": 0,
}

r = requests.post(url, headers=headers, json=payload, timeout=90)

print("HTTP status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("Response body:", repr(r.text[:1000]))

HTTP status: 200
Content-Type: text/event-stream; charset=utf-8
Response body: 'data: {"id":"chatcmpl-b37f982153cfdb9f","object":"chat.completion.chunk","created":1793190659,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"delta":{"role":"assistant","content":""},"logprobs":null,"finish_reason":null}],"prompt_token_ids":null,"prompt_text":null}\n\ndata: {"id":"chatcmpl-b37f982153cfdb9f","object":"chat.completion.chunk","created":1793190659,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"delta":{"content":"Cloud"},"logprobs":null,"finish_reason":null,"token_ids":null}]}\n\ndata: {"id":"chatcmpl-b37f982153cfdb9f","object":"chat.completion.chunk","created":1793190659,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"delta":{"content":" computing"},"logprobs":null,"finish_reason":null,"token_ids":null}]}\n\ndata: {"id":"chatcmpl-b37f982153cfdb9f","object":"chat.completion.chunk","created":1793190659,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0

In [121]:

import requests
import time
import statistics
import json

DIRECT = "http://127.0.0.1:8001/v1/chat/completions"
GATEWAY = "http://127.0.0.1:8000/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {tenant_api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [{
        "role": "user",
        "content": (
            "Explain cloud computing, its architecture, deployment "
            "models, security challenges, and practical applications in detail."
        )
    }],
    "max_tokens": 128,
    "temperature": 0,
}

def measure(url, auth=False):
    request_headers = headers if auth else {
        "Content-Type": "application/json"
    }

    start = time.perf_counter()
    r = requests.post(
        url, headers=request_headers, json=payload,
        timeout=120, stream=True
    )

    try:
        r.raise_for_status()
        content_type = r.headers.get("Content-Type", "")
        token_count = 0
        text_parts = []
        usage_tokens = None

        if "text/event-stream" in content_type:
            for line in r.iter_lines(decode_unicode=True):
                if not line or not line.startswith("data:"):
                    continue

                data = line[5:].strip()
                if data == "[DONE]":
                    break

                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue

                usage = chunk.get("usage") or {}
                if usage.get("completion_tokens") is not None:
                    usage_tokens = usage["completion_tokens"]

                for choice in chunk.get("choices", []):
                    text = choice.get("delta", {}).get("content", "")
                    if text:
                        text_parts.append(text)
        else:
            data = r.json()
            usage_tokens = (data.get("usage") or {}).get("completion_tokens")
            text_parts = [
                choice.get("message", {}).get("content", "")
                for choice in data.get("choices", [])
            ]

        elapsed = time.perf_counter() - start
        generated_text = "".join(text_parts)

        if usage_tokens is not None:
            token_count = usage_tokens
        elif generated_text:
            # Approximation only; characters/4 is NOT an exact token count.
            token_count = None
        else:
            raise RuntimeError(
                f"No generated text or token usage. "
                f"HTTP {r.status_code}: {r.text[:300]}"
            )

        rate = token_count / elapsed if token_count is not None and elapsed else None
        return elapsed, token_count, rate, len(generated_text)

    finally:
        r.close()


# Warm up both paths
print("Warming up...")
print("Direct:", measure(DIRECT))
print("Gateway:", measure(GATEWAY, auth=True))

results = {"Direct": [], "Gateway": []}

for i in range(10):
    order = [
        ("Direct", DIRECT, False),
        ("Gateway", GATEWAY, True),
    ]
    if i % 2:
        order.reverse()

    for name, url, auth in order:
        try:
            elapsed, tokens, rate, chars = measure(url, auth)
            results[name].append((elapsed, tokens, rate, chars))
            print(
                f"Pair {i+1:02d} | {name}: {elapsed:.3f}s, "
                f"tokens={tokens}, throughput={rate}"
            )
        except Exception as e:
            print(f"Pair {i+1:02d} | {name} FAILED: {e}")

for name, rows in results.items():
    print(f"\n--- {name} ---")
    if not rows:
        print("No successful measurements")
        continue

    print(f"Successful requests: {len(rows)}/10")
    print(f"Mean latency: {statistics.mean(x[0] for x in rows):.3f}s")
    print(f"Median latency: {statistics.median(x[0] for x in rows):.3f}s")

    rates = [x[2] for x in rows if x[2] is not None]
    if rates:
        print(f"Mean output throughput: {statistics.mean(rates):.2f} tokens/s")
    else:
        print("Exact token throughput unavailable: API did not report token usage.")


Warming up...
Direct: (2.952340858999378, 128, 43.35542747709098, 684)
Gateway: (14.264341926998895, None, None, 3238)
Pair 01 | Direct: 3.503s, tokens=128, throughput=36.53814225075943
Pair 01 | Gateway: 9.501s, tokens=None, throughput=None
Pair 02 | Gateway: 17.174s, tokens=None, throughput=None
Pair 02 | Direct: 2.552s, tokens=128, throughput=50.15479448588106
Pair 03 | Direct: 2.552s, tokens=128, throughput=50.153690086261534
Pair 03 | Gateway: 14.283s, tokens=None, throughput=None
Pair 04 | Gateway: 13.117s, tokens=None, throughput=None
Pair 04 | Direct: 2.807s, tokens=128, throughput=45.5995287189115
Pair 05 | Direct: 2.868s, tokens=128, throughput=44.6257527884572
Pair 05 | Gateway: 14.934s, tokens=None, throughput=None
Pair 06 | Gateway: 14.135s, tokens=None, throughput=None
Pair 06 | Direct: 2.889s, tokens=128, throughput=44.305912164284095
Pair 07 | Direct: 2.845s, tokens=128, throughput=44.985376951435754
Pair 07 | Gateway: 13.751s, tokens=None, throughput=None
Pair 08 | Gat

In [122]:
from pathlib import Path

log_path = Path("/kaggle/working/gateway.log")

if log_path.exists():
    lines = log_path.read_text(errors="replace").splitlines()
    print("\n".join(lines[-100:]))
else:
    print("Gateway log not found:", log_path)

2026-10-28 12:32:42,034 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
2026-10-28 12:32:47,036 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
INFO:     127.0.0.1:52854 - "POST /v1/chat/completions HTTP/1.1" 200 OK
2026-10-28 12:32:52,040 | INFO | request_id=- | httpx | HTTP Request: GET http://127.0.0.1:8001/health "HTTP/1.1 200 OK"
2026-10-28 12:32:54,085 | INFO | request_id=67f58473-8071-421a-90c9-365a3331eb31 | app.main | Inference request received tenant=kaggle-standard-tenant request_id=67f58473-8071-421a-90c9-365a3331eb31
2026-10-28 12:32:54,085 | INFO | request_id=67f58473-8071-421a-90c9-365a3331eb31 | app.main | Scheduler recommendation tenant=kaggle-standard-tenant backend=kaggle-t4-general zone=Zone.GENERAL
2026-10-28 12:32:54,089 | INFO | request_id=67f58473-8071-421a-90c9-365a3331eb31 | httpx | HTTP Request: POST http://127.0.0.1:8181/v1/data/confidentialkv/authz "HTTP/1.1 20

In [123]:
import requests
import time

url = "http://127.0.0.1:8000/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {tenant_api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [{"role": "user", "content": "Say hello briefly."}],
    "max_tokens": 16,
    "temperature": 0,
}

start = time.perf_counter()
first_chunk = None
chunk_count = 0
body_chars = 0

with requests.post(
    url, headers=headers, json=payload,
    stream=True, timeout=90
) as r:
    print("HTTP:", r.status_code)
    print("Content-Type:", r.headers.get("Content-Type"))
    r.raise_for_status()

    for line in r.iter_lines(decode_unicode=True):
        if line and line.startswith("data:"):
            if first_chunk is None:
                first_chunk = time.perf_counter() - start
            chunk_count += 1
            body_chars += len(line)

elapsed = time.perf_counter() - start

print("Time to first SSE chunk:", first_chunk, "seconds")
print("Total response time:", round(elapsed, 3), "seconds")
print("SSE data lines:", chunk_count)
print("Total SSE line characters:", body_chars)

HTTP: 200
Content-Type: text/event-stream; charset=utf-8
Time to first SSE chunk: 0.22477395700116176 seconds
Total response time: 0.225 seconds
SSE data lines: 12
Total SSE line characters: 2708
